In [ ]:
# Cell 1: Imports and setup
from common_imports import *
from common_utils import evaluate, train_model, setup_data
from torch.optim.lr_scheduler import StepLR

# 2) Model definition (bigger network + BatchNorm + Dropout)
class SimpleCNN(nn.Module):
    def __init__(self, dropout_rate=0.5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=5, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 3 * 3, 256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


def init_weights(m):
    if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)


# 3) Data setup with augmentation
train_transform = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,)),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,)),
])

full_train_dataset = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=train_transform)
val_size = int(len(full_train_dataset) * 0.1)
train_size = len(full_train_dataset) - val_size
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=0, pin_memory=False)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=0, pin_memory=False)
test_loader = DataLoader(torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=test_transform), batch_size=128, shuffle=False, num_workers=0, pin_memory=False)

# 4) Training and model ensemble
NUM_EPOCHS = 15
LEARNING_RATE = 1e-3

models = []
for run in range(2):
    model = SimpleCNN().to('mps' if torch.backends.mps.is_available() else 'cpu')
    model.apply(init_weights)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    scheduler = StepLR(optimizer, step_size=7, gamma=0.5)

    print(f"\n=== Training model {run + 1} ===")
    train_model(model, train_loader, val_loader, criterion, optimizer, model.device if hasattr(model, 'device') else ('mps' if torch.backends.mps.is_available() else 'cpu'), num_epochs=NUM_EPOCHS, patience=4)
    scheduler.step()
    models.append(model)


# 5) Ensemble evaluation

def ensemble_evaluate(models, dataloader, criterion, device):
    for m in models:
        m.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = torch.stack([m(images) for m in models], dim=0)
            avg_logits = outputs.mean(dim=0)
            loss = criterion(avg_logits, labels)

            running_loss += loss.item() * images.size(0)
            preds = avg_logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return running_loss / total, correct / total


device = 'mps' if torch.backends.mps.is_available() else 'cpu'
ensemble_loss, ensemble_acc = ensemble_evaluate(models, test_loader, nn.CrossEntropyLoss(), device)
print(f"\nEnsemble Test Loss: {ensemble_loss:.4f} | Ensemble Test Accuracy: {ensemble_acc:.4f}")


=== Training model 1 ===
